# מפגש 11 — הדגמה מדורגת: רבעונים, Boxplot ואיכות נתונים
הדגמת משלוחים מומצאת. לפני הרצה נבצע **חישוב ידני בשיטת חצאי הרשימה ללא החציון** עבור תשע מדידות, ואז נבדוק שבדוגמה זו התוצאה תואמת גם ל־Pandas linear. השיטות אינן שקולות בכל מערך.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
minutes=[8,10,10,12,14,16,20,20,40]
delivery_df=pd.DataFrame({'minutes':minutes})
print('Sorted minutes:',sorted(minutes))
assert delivery_df.shape==(9,1)

## שלב 1: חישוב רבעונים ביד
החציון הוא 14; ארבעת ערכי החצי התחתון [8,10,10,12] נותנים Q1=10; העליון [16,20,20,40] נותן Q3=20. IQR=10 דקות. מה יתקבל ב־Pandas linear?

In [ ]:
q1=delivery_df['minutes'].quantile(.25,interpolation='linear')
q2=delivery_df['minutes'].median()
q3=delivery_df['minutes'].quantile(.75,interpolation='linear')
iqr=q3-q1
print('Q1',q1,'median',q2,'Q3',q3,'IQR',iqr)
assert (q1,q2,q3,iqr)==(10.,14.,20.,10.)

## שלב 2: גדרות לעומת שפמים
הגדרות −5 ו־35 **אינן** קצות השפמים. הקצוות הנצפים בתוך הגבולות הם 8 ו־20; 40 נקודה שדורשת בירור.

In [ ]:
lower=q1-1.5*iqr
upper=q3+1.5*iqr
known=delivery_df['minutes']
inside=known[(known>=lower)&(known<=upper)]
flagged=known[(known<lower)|(known>upper)]
print('Fences',lower,upper,'Whiskers',inside.min(),inside.max(),'Review',flagged.tolist())
assert (lower,upper)==(-5.,35.)
assert (inside.min(),inside.max())==(8,20)
assert flagged.tolist()==[40]

## שלב 3: המחשת Boxplot — הצירים בדקות
הגרף אינו אומר שהנקודה 40 היא טעות. אין למחוק אותה ללא מידע אמין על מסירת המשלוח.

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=delivery_df,y='minutes',whis=1.5,color='lightblue')
plt.ylabel('Delivery time (minutes)')
plt.title('Nine teaching observations - review 40')
plt.tight_layout()
plt.show()

## שלב 4: שתי שיטות רבעונים יכולות להיות שונות
במערך [2,4,6,8,20] Pandas linear נותנת Q1=4,Q3=8; בשיטת חצאי הרשימה ללא חציון מתקבלים Q1=3,Q3=14. **לא** נחליף שיטה אחרי ראיית התוצאה.

In [ ]:
small=pd.Series([2,4,6,8,20])
linear_q1=small.quantile(.25,interpolation='linear')
linear_q3=small.quantile(.75,interpolation='linear')
exclusive_q1=(2+4)/2
exclusive_q3=(8+20)/2
print('Linear:',linear_q1,linear_q3,'Exclusive halves:',exclusive_q1,exclusive_q3)
assert (linear_q1,linear_q3)==(4.,8.)
assert (exclusive_q1,exclusive_q3)==(3,14)

## שלב 5: זיהוי כפילויות אינו מחיקה
הזמנות 101 ו־102 יכולות לחלוק מחיר וכמות — ועדיין להיות שונות. יש לבדוק מזהה, שדות אחרים ומקור.

In [ ]:
raw=pd.DataFrame({'order_id':[101,102,103,103,104,105],'ordered_at':['2026-01-02','2026-01-03','2026-01-04','2026-01-04','2026-01-05','2026-01-06'],'items':[2,2,1,1,3,2],'unit_price':[10.,10.,20.,20.,15.,10.],'duration_min':[12.,14.,8.,8.,-3.,np.nan]})
work=raw.copy(deep=True)
display(work[work.duplicated(subset=['order_id'],keep=False)])
display(work[work.duplicated(subset=['items','unit_price'],keep=False)])
assert len(raw)==6
assert work.duplicated(subset=['order_id']).sum()==1
assert work.duplicated(subset=['items','unit_price']).sum()>1

## שלב 6: הגנת מקור — זיהוי לפני פעולה
אם order_id אמור להיות ייחודי, בודקים האם שתי הרשומות בעלות אותו מזהה **זהות לחלוטין** או סותרות זו את זו. שינוי מותר רק בעותק עם יומן. מקור raw נשאר ללא שינוי.

In [ ]:
same_id=work[work.duplicated(subset=['order_id'],keep=False)]
print('Rows needing identity review:',len(same_id))
print('Exact duplicated rows:',work.duplicated().sum())
assert len(same_id)==2
assert work.duplicated().sum()==1
assert raw.loc[raw.order_id==104,'duration_min'].iloc[0]==-3

## מעבר למאגר אמיתי — Penguins
בהדגמה נשווה Boxplot של **מסה בגרמים** בין מינים. במעבדת התלמיד ינותח **אורך סנפיר במילימטרים**, כדי להשאיר משימה עצמאית.

In [ ]:
url="https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
penguins_df=pd.read_csv(url)
plt.figure(figsize=(7,4))
sns.boxplot(data=penguins_df,x='species',y='body_mass_g',order=['Adelie','Chinstrap','Gentoo'],whis=1.5,color='lightblue')
plt.xlabel('Species')
plt.ylabel('Body mass (g)')
plt.title('Penguins: mass distribution by species')
plt.tight_layout()
plt.show()
assert penguins_df.shape==(344,7)
assert penguins_df['body_mass_g'].count()==342

## מסקנה עם מגבלה
ה־Boxplot מתאר מרכז ופיזור ואינו מוכיח שגיאה, נורמליות או סיבתיות. בפרויקט התלמיד מתעד מדיניות בדיקה ולא מנקה כדי לשפר ציון.